# 01 · Extracción, limpieza y EDA
### Riesgo de acceso a la salud por municipio en Colombia · Fase 1

Este notebook **narra** la fase 1 paso a paso: de dónde salen los datos, qué problemas
tienen, cómo se corrigen y qué muestra el análisis exploratorio. No copia la lógica:
importa las mismas funciones del paquete `src/riesgo_salud/` que usa el pipeline
(`make pipeline`), así que lo que se ve aquí es exactamente lo que corre en la terminal,
en Docker y en GitHub Actions.

Los parámetros (EPS intervenidas, umbrales, departamento del zoom) están en
`config/config.yaml`. Para cambiarlos se edita ese archivo, no el notebook.

**Cómo abrirlo en VS Code:** abra la carpeta del repo, haga `uv sync` y elija como
kernel el intérprete `.venv` del proyecto.

In [ ]:
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from riesgo_salud import extraccion, features, limpieza, validacion
from riesgo_salud.config import cargar_config, ruta, ultimo_corte

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

cfg = cargar_config()
FECHA = ultimo_corte(cfg) or date.today().isoformat()  # o fije una fecha: FECHA = "2026-10-25"
EPS_INTERVENIDAS = cfg["eps_intervenidas"]["lista"]
print("Carpeta del proyecto:", cfg["rutas"]["base"])
print("Corte:", FECHA)
print("EPS intervenidas:", EPS_INTERVENIDAS)

## 1. Extracción desde la API de datos abiertos

Si el corte ya está en `data/raw/<fecha>/`, se lee de disco; si no, se descargan 7 tablas
de datos.gov.co (`extraccion.extraer`). Así el análisis no depende de que el portal
esté disponible y cada corte queda guardado para compararlo después.

- La BDUA se pide **agregada en el servidor** en dos vistas: por EPS
  (municipio × EPS × régimen × zona × estado) y demográfica (municipio × régimen ×
  sexo × grupo de edad). En vez de ~2 millones de filas llegan ~127 mil.
- Del REPS **no** se piden correos, teléfonos ni documentos (Ley 1581 de 2012).

In [ ]:
raw = extraccion.leer_raw(ruta(cfg, "raw", FECHA)) or extraccion.extraer(cfg, FECHA)
{nombre: len(df) for nombre, df in raw.items()}

> **Resultado del corte 2026-10-25.** Unas 127.000 filas en lugar de casi 2 millones: BDUA por
> EPS 28.301 (contributivo) y 25.333 (subsidiado), BDUA demográfica 26.102 y 26.942,
> REPS 19.551 sedes de IPS y DIVIPOLA 1.122 municipios.

## 2. Perfilamiento de calidad
Antes de transformar nada se revisa cada tabla: tipos, nulos, duplicados y cardinalidad.

In [ ]:
def perfil(df, nombre):
    p = pd.DataFrame(
        {
            "tipo": df.dtypes.astype(str),
            "nulos": df.isna().sum(),
            "pct_nulos": (df.isna().mean() * 100).round(2),
            "valores_unicos": df.nunique(),
            "ejemplo": df.iloc[0] if len(df) else None,
        }
    )
    print(
        f"\n=== {nombre}: {len(df):,} filas · {df.shape[1]} columnas · duplicados: {df.duplicated().sum():,}"
    )
    display(p)


for nombre, df in raw.items():
    perfil(df, nombre)

In [ ]:
# Valores de las categóricas clave: aquí aparecen las inconsistencias a corregir
eps_crudo = pd.concat([raw["eps_contributivo"], raw["eps_subsidiado"]])
for col in ["tps_rgm_nombre", "zns_nombre", "tps_est_afl_nombre"]:
    print(col, "→", sorted(eps_crudo[col].dropna().unique()))

# Control de totales: deben cuadrar con lo publicado (~23 M contributivo, ~27 M subsidiado)
for k in ["eps_contributivo", "eps_subsidiado", "demo_contributivo", "demo_subsidiado"]:
    print(f"{k:<20} total afiliados: {pd.to_numeric(raw[k]['afiliados']).sum():>14,.0f}")

> **Lo que se encontró en el corte 2026-10-25**
>
> - Ninguna tabla tiene nulos ni filas duplicadas.
> - `afiliados` llega como texto: se convierte a número en la limpieza.
> - DIVIPOLA tiene 1.122 municipios pero solo 1.037 nombres distintos (varios
>   "San Pedro" o "Bolívar"): la unión debe ser por **departamento + municipio**.
> - Las coordenadas de DIVIPOLA usan coma decimal (`6,246631`).
> - La zona tiene 6 nombres distintos y el estado del afiliado dos ("Activo" y
>   "Protección Laboral", ambos con cobertura, así que se cuentan los dos).

## 3. Limpieza y normalización
### 3.1 Normalización de texto
Mayúsculas, sin tildes ni signos, y corrección del error `¥` por `Ñ` de la BDUA.

In [ ]:
for ejemplo in ["NARI¥O", "Bogotá, D.C.", "  la paz  robles "]:
    print(f"{ejemplo!r:<22} → {limpieza.normalizar_texto(ejemplo)!r}")

### 3.2 Homologación de municipios con el código DIVIPOLA
El REPS ya trae el código DIVIPOLA, pero la BDUA solo trae nombres. El cruce
(`limpieza.asignar_divipola`) se hace en este orden:

1. **Tabla de homologación manual** (`HOMOLOGACION_MANUAL` en `limpieza.py`), revisada
   uno a uno: `CUCUTA` → `SAN JOSE DE CUCUTA`, `BUGA` → `GUADALAJARA DE BUGA`,
   `SAN PEDRO` (Antioquia) → `SAN PEDRO DE LOS MILAGROS`...
2. **Cruce exacto** por departamento + municipio, con alias de departamento
   (`VALLE` → `VALLE DEL CAUCA`).
3. `DPTO. GUAINIA` y `DPTO. VAUPES` son **áreas no municipalizadas**: se reportan y
   quedan fuera de la tabla municipal.

In [ ]:
div = limpieza.preparar_divipola(raw["divipola"])
bdua_eps, bdua_demo, cruce = limpieza.limpiar_bdua(raw, div, EPS_INTERVENIDAS)
display(cruce)

pendientes = bdua_eps.loc[bdua_eps["estado_cruce"] == "sin_cruce", ["dpr_nombre", "mnc_nombre"]]
if len(pendientes):
    print("Municipios sin cruce. Agréguelos a HOMOLOGACION_MANUAL en limpieza.py:")
    display(pendientes.drop_duplicates())
else:
    print("Todos los municipios de la BDUA quedaron homologados con DIVIPOLA.")

> **Resultado del corte 2026-10-25.** 987 municipios cruzaron por nombre exacto (46,5 millones de
> afiliados, 92,8 %) y 54 por homologación manual (3,6 millones, 7,2 %). Sin la
> homologación se habrían perdido Cúcuta, Cartagena, Tumaco y Buga.

### 3.3 Unificación de EPS y marcación de intervenidas
Una EPS aparece con varios códigos: uno por régimen y otro con `-CM` (movilidad).
`limpieza.eps_canonica` les asigna un nombre corto con las reglas de `REGLAS_EPS`.

In [ ]:
resumen_eps = limpieza.resumen_eps(bdua_eps)
resumen_eps

> **Resultado del corte 2026-10-25.** Unos 60 códigos quedaron en 28 EPS. Nueva EPS es la más grande
> (11,3 millones, 22,5 %) y cuatro EPS (Nueva EPS, Sanitas, Sura y Salud Total)
> concentran el 57 % de los afiliados.

### 3.4 REPS y validaciones de calidad
Del REPS se usan solo las sedes de clase **IPS**: los 54.657 profesionales
independientes inflarían la oferta hospitalaria. Luego se corren los **contratos de
datos** (`validacion.py`). Si un chequeo crítico falla, el pipeline se detiene y no se
entrena ningún modelo con datos corruptos.

In [ ]:
display(
    raw["reps_resumen"]
    .assign(sedes=lambda d: pd.to_numeric(d["sedes"]))
    .pivot_table(
        index="claseprestador", columns="naturalezajuridica", values="sedes", aggfunc="sum", fill_value=0
    )
)

reps, info_reps = limpieza.limpiar_reps(raw, div)
print(
    f"Sedes de IPS: {len(reps):,} · duplicadas: {info_reps['duplicados']} · "
    f"códigos de municipio inexistentes: {info_reps['sedes_sin_divipola']}\n"
)

validacion.aplicar(validacion.validar_preparacion(bdua_eps, bdua_demo, reps, div, cruce, cfg))

> **Resultado del corte 2026-10-25.** Todos los chequeos en OK con 50.137.787 afiliados: la limpieza no
> perdió ni duplicó personas. El REPS quedó con 19.547 sedes de IPS (se excluyeron 4
> con el código 27086, que no existe en DIVIPOLA).

## 4. Tabla analítica municipal
Una fila por municipio con los indicadores de **dependencia** (BDUA) y de **oferta**
(REPS). El diccionario de columnas está en `docs/diccionario_datos.md`.

**HHI**: se eleva al cuadrado la participación (en %) de cada EPS y se suma. Con 60 % y
40 %: 60² + 40² = 5.200. Con una sola EPS: 10.000. Más de 2.500 es "muy concentrado".

In [ ]:
tabla = features.construir_tabla_municipal(bdua_eps, reps, div)
total_municipal = bdua_eps.loc[bdua_eps["cod_mpio"].notna(), "afiliados"].sum()
validacion.aplicar(validacion.validar_tabla_municipal(tabla, total_municipal))
tm = tabla.dropna(subset=["afiliados"]).copy()
display(tm.describe().T)

> **Resultado del corte 2026-10-25.** 1.122 municipios, todos con afiliados y con al menos una sede de
> IPS. La mediana es de 10.320 afiliados, 86 % subsidiado, HHI de 5.188 y 2,87 sedes de
> IPS por cada 10.000 afiliados.

## 5. Análisis exploratorio (EDA)

In [ ]:
AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8c8b86"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.titleweight": "bold",
        "font.size": 10,
    }
)

umbral = cfg["eda"]["umbral_dependencia"]
pct_int_nac = bdua_eps.loc[bdua_eps["eps_intervenida"], "afiliados"].sum() / bdua_eps["afiliados"].sum()
print(f"% nacional en EPS intervenidas: {pct_int_nac:.1%}")
print(
    f"Municipios con ≥ {umbral:.0%} en EPS intervenidas: {(tm['pct_intervenidas'] >= umbral).sum()} de {len(tm)}"
)

> **Resultado del corte 2026-10-25.** El 42,3 % de los afiliados (21,2 millones) está en una EPS
> intervenida y en 702 de 1.122 municipios (63 %) la mitad o más depende de ellas.

### 5.1 ¿Quién concentra la afiliación? Top 15 EPS

In [ ]:
top = resumen_eps.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(top["eps"], top["afiliados"] / 1e6, color=np.where(top["eps_intervenida"], NARANJA, AZUL), height=0.7)
for y, v in enumerate(top["afiliados"] / 1e6):
    ax.text(
        v + top["afiliados"].max() / 1e6 * 0.01, y, f"{v:.1f} M", va="center", fontsize=8.5, color="#52514e"
    )
ax.set_xlabel("Afiliados (millones)")
ax.set_title("Top 15 EPS por afiliados (contributivo + subsidiado)")
ax.legend(
    handles=[plt.Rectangle((0, 0), 1, 1, color=NARANJA), plt.Rectangle((0, 0), 1, 1, color=AZUL)],
    labels=["Intervenida", "No intervenida"],
    loc="lower right",
    frameon=False,
)
ax.grid(axis="y", visible=False)
plt.show()

> **Resultado del corte 2026-10-25.** Nueva EPS tiene casi el doble de afiliados que Sanitas. De las
> 15 EPS más grandes, 6 están intervenidas: la crisis incluye a la más grande del país.

### 5.2 Distribución de los indicadores municipales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(tm["pct_intervenidas"] * 100, bins=30, color=AZUL, edgecolor="white")
axes[0].set(title="% afiliados en EPS intervenidas", xlabel="%", ylabel="Municipios")
axes[1].hist(tm["hhi"], bins=30, color=AZUL, edgecolor="white")
axes[1].axvline(2500, color=GRIS, ls="--", lw=1)
axes[1].text(2600, axes[1].get_ylim()[1] * 0.9, "muy concentrado", fontsize=8)
axes[1].set(title="Concentración de EPS (HHI)", xlabel="HHI")
axes[2].hist(
    tm["ips_x_10mil"].clip(upper=tm["ips_x_10mil"].quantile(0.99)), bins=30, color=AZUL, edgecolor="white"
)
axes[2].set(title="Sedes de IPS por 10.000 afiliados", xlabel="Sedes / 10.000 (recortado al p99)")
plt.tight_layout()
plt.show()

> **Resultado del corte 2026-10-25.** El % en EPS intervenidas tiene forma de "J": en unos 260 municipios
> casi el 100 % de la población depende de ellas. Casi todos superan el HHI de 2.500 y
> la mayoría tiene entre 1 y 5 sedes de IPS por cada 10.000 afiliados.

### 5.3 Estructura poblacional por régimen

In [ ]:
edad = pd.Categorical(bdua_demo["grupo_edad"], categories=limpieza.ORDEN_EDAD, ordered=True)
pir = (
    bdua_demo.assign(grupo_edad=edad)
    .groupby(["grupo_edad", "regimen"], observed=True)["afiliados"]
    .sum()
    .unstack()
    / 1e6
)
fig, ax = plt.subplots(figsize=(9, 4.2))
x, w = np.arange(len(pir)), 0.4
ax.bar(x - w / 2, pir["CONTRIBUTIVO"], w - 0.03, color=AZUL, label="Contributivo")
ax.bar(x + w / 2, pir["SUBSIDIADO"], w - 0.03, color=NARANJA, label="Subsidiado")
ax.set_xticks(x, pir.index.astype(str), fontsize=8.5)
ax.set(ylabel="Afiliados (millones)", title="Afiliados por grupo de edad y régimen")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
plt.show()

> **Resultado del corte 2026-10-25.** El grupo más grande en ambos regímenes es el de 19 a 45 años. El
> subsidiado tiene 4,3 millones de niños de 5 a 15 años frente a 2,5 millones en el
> contributivo: si una EPS del subsidiado falla, afecta a muchos niños.

### 5.4 Relación entre indicadores (Spearman)

In [ ]:
vars_corr = [
    "pct_intervenidas",
    "hhi",
    "pct_eps_principal",
    "n_eps",
    "ips_x_10mil",
    "pct_rural",
    "pct_subsidiado",
]
corr = tm[vars_corr].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(vars_corr)), vars_corr, rotation=45, ha="right")
ax.set_yticks(range(len(vars_corr)), vars_corr)
for i in range(len(vars_corr)):
    for j in range(len(vars_corr)):
        ax.text(
            j,
            i,
            f"{corr.iat[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color="white" if abs(corr.iat[i, j]) > 0.6 else "#0b0b0b",
        )
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman")
ax.set_title("Correlación entre indicadores municipales")
plt.show()

> **Resultado del corte 2026-10-25.** La dependencia de EPS intervenidas casi no se relaciona con la
> oferta de IPS (0,02): son dos dimensiones distintas y el índice de riesgo necesita
> ambas. El HHI y el % de la EPS principal miden casi lo mismo (0,98): por eso el
> clustering usa solo el HHI. Los municipios más rurales son más subsidiados (0,59) y
> tienen menos EPS (−0,53).

### 5.5 El cruce clave: dependencia de EPS intervenidas vs. oferta de IPS
Cada punto es un municipio y su tamaño es proporcional a los afiliados. El cuadrante
superior izquierdo (**alta dependencia y baja oferta**) es donde está el riesgo. Esta
clasificación (`features.clasificar_cuadrante`) es exploratoria: en la fase 2 la
reemplaza el índice de riesgo (notebook 02).

El eje horizontal usa escala logarítmica (todos los municipios tienen al menos una
sede, así que no hay ceros).

In [ ]:
tm["cuadrante"] = features.clasificar_cuadrante(tm, umbral)
mediana_oferta = tm["ips_x_10mil"].median()
riesgo = tm["cuadrante"] == features.CUADRANTES[0]
tam = np.sqrt(tm["afiliados"]) / 8

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    tm.loc[~riesgo, "ips_x_10mil"],
    tm.loc[~riesgo, "pct_intervenidas"] * 100,
    s=tam[~riesgo],
    color=AZUL,
    alpha=0.45,
    edgecolor="white",
    linewidth=0.5,
    label="Resto de municipios",
)
ax.scatter(
    tm.loc[riesgo, "ips_x_10mil"],
    tm.loc[riesgo, "pct_intervenidas"] * 100,
    s=tam[riesgo],
    color=NARANJA,
    alpha=0.8,
    edgecolor="white",
    linewidth=0.5,
    label=features.CUADRANTES[0],
)
ax.axhline(umbral * 100, color=GRIS, ls="--", lw=1)
ax.axvline(mediana_oferta, color=GRIS, ls="--", lw=1)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))  # 1, 10, 100 en vez de 10¹
ax.set(
    xlabel="Sedes de IPS por 10.000 afiliados (escala log)",
    ylabel="% afiliados en EPS intervenidas",
    title="Dependencia de EPS intervenidas vs. oferta de IPS por municipio",
)
for _, r in tm[riesgo].nlargest(8, "afiliados").iterrows():
    ax.annotate(
        r["nom_mpio"].title(),
        (r["ips_x_10mil"], r["pct_intervenidas"] * 100),
        fontsize=7.5,
        xytext=(4, 4),
        textcoords="offset points",
    )
ax.legend(frameon=True, framealpha=0.9, loc="best", markerscale=0.6)
plt.show()

tm.groupby("cuadrante").agg(municipios=("cod_mpio", "size"), afiliados=("afiliados", "sum")).reindex(
    features.CUADRANTES
)

> **Resultado del corte 2026-10-25.** 
>
> | Grupo | Municipios | Afiliados |
> |---|---|---|
> | **Alta dependencia · baja oferta (riesgo)** | **362** | **6.627.358** |
> | Alta dependencia · oferta media/alta | 340 | 9.704.525 |
> | Baja dependencia · baja oferta | 199 | 6.690.539 |
> | Baja dependencia · oferta media/alta | 221 | 27.111.599 |
>
> 362 municipios con 6,6 millones de personas (13 % de los afiliados) están en la zona
> de mayor riesgo. La mediana nacional de oferta es 2,87 sedes por 10.000.

### 5.6 Mapa exploratorio

In [ ]:
import plotly.express as px

mapa = tm.dropna(subset=["latitud", "longitud"]).assign(
    pct_int=lambda d: (d["pct_intervenidas"] * 100).round(1)
)
kw = dict(
    lat="latitud",
    lon="longitud",
    color="pct_int",
    size="afiliados",
    size_max=28,
    hover_name="nom_mpio",
    hover_data={
        "dpto": True,
        "eps_principal": True,
        "sedes_ips": True,
        "ips_x_10mil": ":.1f",
        "latitud": False,
        "longitud": False,
    },
    color_continuous_scale="Oranges",
    range_color=(0, 100),
    zoom=4.3,
    center={"lat": 4.6, "lon": -74.1},
    height=650,
    labels={"pct_int": "% en EPS intervenidas"},
    title="% de afiliados en EPS intervenidas por municipio (tamaño = afiliados)",
)
try:
    fig = px.scatter_map(mapa, map_style="carto-positron", **kw)
except AttributeError:  # versiones antiguas de plotly
    fig = px.scatter_mapbox(mapa, mapbox_style="carto-positron", **kw)
fig.show()

> **Resultado del corte 2026-10-25.** Mayor dependencia (promedio ponderado por afiliados): Guaviare y
> Guainía (100 %), Caquetá (90 %), Arauca (85 %), Vaupés (84 %), Putumayo (79 %).
> Menor: La Guajira (14 %), Antioquia (20 %) y Bogotá (22 %).

### 5.7 Ranking preliminar y zoom departamental

In [ ]:
cols_rank = [
    "dpto",
    "nom_mpio",
    "afiliados",
    "eps_principal",
    "pct_eps_principal",
    "pct_intervenidas",
    "sedes_ips",
    "ips_x_10mil",
    "pct_rural",
]
print("Municipios en el cuadrante de riesgo, por población afiliada (Top 20 nacional):")
display(tm[riesgo].nlargest(20, "afiliados")[cols_rank])

depto = cfg["eda"]["depto_foco"]
if depto:
    foco = tm[tm["dpto"].map(limpieza.normalizar_texto) == limpieza.normalizar_texto(depto)]
    pct = foco["pct_intervenidas"].mul(foco["afiliados"]).sum() / foco["afiliados"].sum()
    print(
        f"\n{depto}: {len(foco)} municipios · {foco['afiliados'].sum():,.0f} afiliados · {pct:.1%} en EPS intervenidas"
    )
    display(foco["eps_principal"].value_counts().rename("municipios donde es la EPS principal").to_frame())
    display(
        foco.sort_values(["pct_intervenidas", "ips_x_10mil"], ascending=[False, True]).head(15)[cols_rank]
    )

> **Resultado del corte 2026-10-25.** Los municipios de mayor riesgo con más población son Jamundí,
> Girón y Villa del Rosario; en 12 de los 20 primeros la EPS principal es Nueva EPS.
> Muchos están junto a una ciudad grande cuyas IPS probablemente usan.
>
> En Antioquia solo el 19,9 % está en EPS intervenidas porque Savia Salud (EPS principal
> en 73 de 125 municipios) no se marcó como intervenida. El riesgo aparece en el Bajo
> Cauca y el Nordeste, que dependen de Coosalud: Nechí (94 % y 0,71 sedes por 10.000),
> El Bagre y Cáceres. Para el escenario con Savia Salud intervenida, agréguela a
> `eps_intervenidas.lista` en `config/config.yaml`.

## 6. Hallazgos y limitaciones

**Hallazgos**
- 42,3 % de los afiliados del país está en una EPS intervenida; en 702 de 1.122 municipios es la mitad o más.
- 362 municipios con 6,6 millones de afiliados combinan alta dependencia y baja oferta de IPS.
- Ningún municipio está totalmente sin IPS, pero la mediana es de 3 sedes por municipio.
- La afiliación está muy concentrada (mediana del HHI 5.188).
- Dependencia y oferta no están correlacionadas: el índice de riesgo debe combinar ambas.
- El riesgo se concentra en la Orinoquía, la Amazonía, el sur del país y municipios intermedios junto a grandes ciudades.

**Limitaciones**
1. **Datos agregados.** La BDUA no tiene registros por paciente: el análisis es municipal.
2. **Oferta aproximada.** El REPS cuenta sedes, no capacidad, ni qué IPS contrata con qué EPS.
3. **Municipios vecinos.** Solo se cuentan las IPS dentro del municipio.
4. **Cortes distintos.** BDUA de octubre de 2026 y REPS de marzo de 2026.
5. **La lista de EPS intervenidas cambia.** La decisión sobre Savia Salud cambia por completo Antioquia.
6. **Sin histórico en el portal.** Las tendencias salen de acumular cortes en `data/raw/<fecha>/`.

Siguiente notebook: **`02_indice_clustering_mlops.ipynb`**.